In [32]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re
import unicodedata


from renewables_permitting.utils import as_list

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Funciones de parseo

In [3]:
def parse_item(
    item: dict[str, Any],
    fecha_publicacion: datetime,
    diario_numero: str | None,
    seccion_codigo: str | None,
    seccion_nombre: str | None,
    departamento_codigo: str | None,
    departamento_nombre: str | None,
    epigrafe_nombre: str | None,
    item_location: str,
) -> dict[str, Any]:
    """
    Transforma un item del sumario BOE en un registro tabular normalizado.
    """
    url_pdf = item.get("url_pdf")

    if isinstance(url_pdf, dict):
        pdf_link = url_pdf.get("texto")
        pdf_size_bytes = url_pdf.get("szBytes")
        pdf_size_kbytes = url_pdf.get("szKBytes")
        pagina_inicial = url_pdf.get("pagina_inicial")
        pagina_final = url_pdf.get("pagina_final")
    else:
        pdf_link = url_pdf
        pdf_size_bytes = None
        pdf_size_kbytes = None
        pagina_inicial = None
        pagina_final = None

    return {
        "identificador": item.get("identificador"),
        "control": item.get("control"),
        "titulo": item.get("titulo"),
        "url_html": item.get("url_html"),
        "url_xml": item.get("url_xml"),
        "url_pdf": pdf_link,
        "pdf_size_bytes": pdf_size_bytes,
        "pdf_size_kbytes": pdf_size_kbytes,
        "pagina_inicial": pagina_inicial,
        "pagina_final": pagina_final,
        "fecha_publicacion": fecha_publicacion.date().isoformat(),
        "year": fecha_publicacion.year,
        "month": fecha_publicacion.month,
        "day": fecha_publicacion.day,
        "diario_numero": diario_numero,
        "seccion_codigo": seccion_codigo,
        "seccion_nombre": seccion_nombre,
        "departamento_codigo": departamento_codigo,
        "departamento_nombre": departamento_nombre,
        "epigrafe_nombre": epigrafe_nombre,
        "item_location": item_location,
        "source": "boe",
        "country": "ES",
    }

In [4]:
def parse_sumario_boe(data: dict[str, Any]) -> pd.DataFrame:
    """
    Convierte la respuesta JSON de la API de Sumarios del BOE en un DataFrame.

    Extrae items ubicados en:
    - departamento.epigrafe.item
    - departamento.item
    - departamento.texto.item
    """
    rows: list[dict[str, Any]] = []

    try:
        sumario = data["data"]["sumario"]
    except KeyError as exc:
        raise ValueError(
            "La respuesta no contiene la estructura esperada: data.sumario."
        ) from exc

    fecha = sumario.get("metadatos", {}).get("fecha_publicacion")

    if not fecha:
        raise ValueError(
            "La respuesta no contiene 'fecha_publicacion' en los metadatos."
        )

    try:
        dt = datetime.strptime(fecha, "%Y%m%d")
    except ValueError as exc:
        raise ValueError(
            f"La fecha_publicacion no tiene formato AAAAMMDD válido: {fecha}"
        ) from exc

    for diario in as_list(sumario.get("diario")):
        diario_numero = diario.get("numero")

        for seccion in as_list(diario.get("seccion")):
            seccion_codigo = seccion.get("codigo")
            seccion_nombre = seccion.get("nombre")

            for departamento in as_list(seccion.get("departamento")):
                departamento_codigo = departamento.get("codigo")
                departamento_nombre = departamento.get("nombre")

                for epigrafe in as_list(departamento.get("epigrafe")):
                    epigrafe_nombre = epigrafe.get("nombre")

                    for item in as_list(epigrafe.get("item")):
                        rows.append(
                            parse_item(
                                item=item,
                                fecha_publicacion=dt,
                                diario_numero=diario_numero,
                                seccion_codigo=seccion_codigo,
                                seccion_nombre=seccion_nombre,
                                departamento_codigo=departamento_codigo,
                                departamento_nombre=departamento_nombre,
                                epigrafe_nombre=epigrafe_nombre,
                                item_location="departamento.epigrafe.item",
                            )
                        )

                for item in as_list(departamento.get("item")):
                    rows.append(
                        parse_item(
                            item=item,
                            fecha_publicacion=dt,
                            diario_numero=diario_numero,
                            seccion_codigo=seccion_codigo,
                            seccion_nombre=seccion_nombre,
                            departamento_codigo=departamento_codigo,
                            departamento_nombre=departamento_nombre,
                            epigrafe_nombre=None,
                            item_location="departamento.item",
                        )
                    )

                texto = departamento.get("texto")
                if isinstance(texto, dict):
                    for item in as_list(texto.get("item")):
                        rows.append(
                            parse_item(
                                item=item,
                                fecha_publicacion=dt,
                                diario_numero=diario_numero,
                                seccion_codigo=seccion_codigo,
                                seccion_nombre=seccion_nombre,
                                departamento_codigo=departamento_codigo,
                                departamento_nombre=departamento_nombre,
                                epigrafe_nombre=None,
                                item_location="departamento.texto.item",
                            )
                        )

    return pd.DataFrame(rows)

In [5]:
fecha = "20260501"

path = BRONZE_DIR / "boe" / fecha / "sumario.json"

with open(path, encoding="utf-8") as f:
    data = json.load(f)

# Funciones para tabla acumulativa parquet

In [6]:
def load_sumario_json(path: Path) -> dict[str, Any]:
    """
    Carga un archivo sumario.json desde disco.
    """
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

In [7]:
def get_fecha_from_sumario_path(path: Path) -> str:
    """
    Extrae la fecha YYYYMMDD desde una ruta tipo:

        data/bronze/boe/20260501/sumario.json
    """
    return path.parent.name

In [8]:
def load_existing_boe_items(
    output_path: Path = SILVER_DIR / "boe_items" / "boe_items.parquet",
) -> pd.DataFrame:
    """
    Carga la tabla silver boe_items si existe.
    Si no existe, devuelve un DataFrame vacío.
    """
    if output_path.exists():
        return pd.read_parquet(output_path)

    return pd.DataFrame()

In [9]:
def parse_new_bronze_sumarios(
    bronze_boe_dir: Path = BRONZE_DIR / "boe",
    output_path: Path = SILVER_DIR / "boe_items" / "boe_items.parquet",
) -> pd.DataFrame:
    """
    Parsea únicamente los sumario.json de bronze que todavía no están
    incorporados en la tabla silver boe_items.

    La tabla silver se actualiza de forma acumulativa.
    """
    existing_df = load_existing_boe_items(output_path)

    if existing_df.empty:
        processed_dates: set[str] = set()
    else:
        processed_dates = set(
            pd.to_datetime(existing_df["fecha_publicacion"])
            .dt.strftime("%Y%m%d")
            .dropna()
            .unique()
        )

    json_files = sorted(bronze_boe_dir.glob("*/sumario.json"))

    pending_files = [
        path
        for path in json_files
        if get_fecha_from_sumario_path(path) not in processed_dates
    ]

    dfs: list[pd.DataFrame] = []

    for json_file in pending_files:
        try:
            data = load_sumario_json(json_file)
            df_day = parse_sumario_boe(data)

            if not df_day.empty:
                df_day["bronze_path"] = str(json_file)
                df_day["bronze_date"] = get_fecha_from_sumario_path(json_file)
                dfs.append(df_day)

        except Exception as exc:
            print(f"Error parseando {json_file}: {exc}")

    if not dfs:
        print("No hay nuevos sumarios pendientes de parsear.")
        return existing_df

    new_df = pd.concat(dfs, ignore_index=True)

    if existing_df.empty:
        final_df = new_df
    else:
        final_df = pd.concat([existing_df, new_df], ignore_index=True)

    final_df = (
        final_df
        .drop_duplicates(subset=["identificador"], keep="last")
        .sort_values(["fecha_publicacion", "identificador"])
        .reset_index(drop=True)
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_parquet(output_path, index=False)

    print(f"Sumarios nuevos procesados: {len(pending_files)}")
    print(f"Filas nuevas añadidas: {len(new_df)}")
    print(f"Filas totales boe_items: {len(final_df)}")

    return final_df

# Parseo de prueba

In [20]:
boe_items = parse_new_bronze_sumarios()

Sumarios nuevos procesados: 1
Filas nuevas añadidas: 193
Filas totales boe_items: 9416


In [30]:
boe_items = pd.read_parquet(SILVER_DIR / "boe_items" / "boe_items.parquet")
boe_items.head(5)
boe_items.columns

Index(['identificador', 'control', 'titulo', 'url_html', 'url_xml', 'url_pdf',
       'pdf_size_bytes', 'pdf_size_kbytes', 'pagina_inicial', 'pagina_final',
       'fecha_publicacion', 'year', 'month', 'day', 'diario_numero',
       'seccion_codigo', 'seccion_nombre', 'departamento_codigo',
       'departamento_nombre', 'epigrafe_nombre', 'item_location', 'source',
       'country', 'bronze_path', 'bronze_date'],
      dtype='object')

# Funciones normalización robusta

In [27]:
def normalize_text(text: str) -> str:

    if pd.isna(text):
        return ""

    text = (
        str(text)
        .lower()
    )

    text = (
        unicodedata.normalize("NFKD", text)
        .encode("ascii", errors="ignore")
        .decode("utf-8")
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Normalización de prueba

In [ ]:
boe_items = pd.read_parquet(SILVER_DIR / "boe_items" / "boe_items.parquet")

boe_items["titulo_norm"] = boe_items["titulo"].apply(normalize_text)
boe_items["departamento_nombre_norm"] = boe_items["departamento_nombre"].apply(normalize_text)
boe_items["epigrafe_nombre_norm"] = boe_items["epigrafe_nombre"].apply(normalize_text)

# Nueva columna de texto normalizado para búsquedas, combinando título, departamento y epígrafe
boe_items["search_text_norm"] = (
    boe_items["titulo_norm"]
    + " "
    + boe_items["departamento_nombre_norm"]
    + " "
    + boe_items["epigrafe_nombre_norm"]
)

In [36]:
boe_items.to_parquet(SILVER_DIR / "boe_items" / "boe_items_normalized.parquet", index=False)